# Deadtime explorer (packet threshold)

Interactive views for the double-pulse deadtime scans.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, Dropdown, Checkbox, IntSlider
from IPython.display import clear_output
import sys

sns.set_context('talk')

SELECT = "(select)"

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT / 'src'))
from deadtime_analysis import DeadtimeAnalysis

DATA_FILES = [
    ROOT / 'data' / 'double_pulse_deadtime-01-14-26.jsonl',
]

analysis_full = DeadtimeAnalysis.from_jsonl([str(p) for p in DATA_FILES])

est_df = None
EST_PATH = ROOT / 'data' / 'estimated_deadtime_01-14-26_packet.json'
if EST_PATH.exists():
    with EST_PATH.open() as fh:
        est_df = pd.DataFrame(json.load(fh))
print('Loaded', len(analysis_full.df), 'rows from', len(DATA_FILES), 'files')
if est_df is not None:
    print('Estimates rows:', len(est_df))


Loaded 14274 rows from 1 files
Estimates rows: 793


## Interactive packet rate vs separation

In [2]:
pulse_rate_options = [SELECT] + sorted(analysis_full.df['pulse_rate_hz'].dropna().unique())
pulse_options = [SELECT] + sorted(analysis_full.df['num_pulses'].dropna().unique())
channel_options = [SELECT] + ['(all)'] + sorted(analysis_full.df['channel_count'].dropna().unique())
window_options = [SELECT] + ['(all)'] + sorted(analysis_full.df['windows'].dropna().unique())

y_min = None
y_max = None

def _target_ratio():
    return 2.0

def _tolerance():
    return 0.01

@interact(
    pulse_rate=Dropdown(options=pulse_rate_options, value=SELECT, description='Pulse Rate (Hz)'),
    num_pulses=Dropdown(options=pulse_options, value=SELECT, description='Pulses'),
    channels=Dropdown(options=channel_options, value=SELECT, description='Channels'),
    windows=Dropdown(options=window_options, value=SELECT, description='Windows'),
    show_expected=Checkbox(value=True, description='Show expected'),
    show_target=Checkbox(value=True, description='Show target (2 - tol)'),
)
def _plot_packet_rates(pulse_rate, num_pulses, channels, windows, show_expected, show_target):
    clear_output(wait=True)
    if pulse_rate in (None, SELECT) or num_pulses in (None, SELECT) or channels in (None, SELECT) or windows in (None, SELECT):
        return

    df = analysis_full.df[analysis_full.df['pulse_rate_hz'] == pulse_rate].copy()
    df = df[df['num_pulses'] == num_pulses]

    ch_val = None if channels == '(all)' else channels
    win_val = None if windows == '(all)' else windows
    if ch_val is not None:
        df = df[df['channel_count'] == ch_val]
    if win_val is not None:
        df = df[df['windows'] == win_val]
    df = df.dropna(subset=['observed_packets_per_sec'])
    if df.empty:
        print('No packet-rate data for selection')
        return

    ana = DeadtimeAnalysis(df, single_factor=analysis_full.single_factor, double_factor=analysis_full.double_factor)
    ana.plot_packet_rate_vs_separation_by_channels(
        pulse_rate,
        num_pulses=num_pulses,
        show_expected=show_expected,
        show_target=show_target,
        target_ratio=_target_ratio(),
        tolerance=_tolerance(),
        y_min=y_min,
        y_max=y_max,
    )
    ana.plot_packet_rate_vs_separation_by_windows(
        pulse_rate,
        num_pulses=num_pulses,
        show_expected=show_expected,
        show_target=show_target,
        target_ratio=_target_ratio(),
        tolerance=_tolerance(),
        y_min=y_min,
        y_max=y_max,
    )


interactive(children=(Dropdown(description='Pulse Rate (Hz)', options=('(select)', np.float64(100.0)), value='…